source API URL : "https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum"

JSON Target File Path : "abfss://bronze@datalakestorageaccountname.dfs.core.windows.net/weather-data/
"

In [0]:
#weatherDataSourceAPIURL = "https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum"

weatherDataSourceAPIBaseURL = "https://archive-api.open-meteo.com/v1/archive?latitude="
weatherDataSourceAPIURLOptions = "&daily=temperature_2m_max,temperature_2m_min,rain_sum"

weatherDataSinkLayerName = 'bronze-dev'
weatherDataSinkStorageAccountName = 'dbrkcrse20251storagedev'
weatherDataSinkFolderName = 'weather-data'

weatherDataSinkFolderPath = f"abfss://{weatherDataSinkLayerName}@{weatherDataSinkStorageAccountName}.dfs.core.windows.net/{weatherDataSinkFolderName}"

In [0]:
import requests
import json
import pandas as pds

In [0]:
geoLocationsDF = spark.sql("SELECT latitude,longitude,marketName from `pricing-analytics`.silver.geo_location_silver limit 100")
display(geoLocationsDF.count())

In [0]:

weatherDataAPIResponseList = []
for geoLocations in geoLocationsDF.collect():
  #print(geoLocations["marketName"],geoLocations["latitude"],geoLocations["longitude"])
  weatherDataSourceAPIURL = f"{weatherDataSourceAPIBaseURL}{geoLocations['latitude']}&longitude={geoLocations['longitude']}&start_date=2023-01-01&end_date=2023-12-31{weatherDataSourceAPIURLOptions}"
  weatherDataAPIResponse = requests.get(weatherDataSourceAPIURL).json()
  weatherDataAPIResponse["marketName"] = geoLocations["marketName"]
  weatherDataAPIResponseJson = json.dumps(weatherDataAPIResponse)
  #print(weatherDataAPIResponseJson)
  if isinstance(weatherDataAPIResponse, dict):
   weatherDataAPIResponseList.append(weatherDataAPIResponseJson)

df = spark.createDataFrame(weatherDataAPIResponseList, "string").toDF("json")
df.printSchema()
display(df)

from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

schema = StructType([
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("generationtime_ms", DoubleType()),
    StructField("utc_offset_seconds", LongType()),
    StructField("timezone", StringType()),
    StructField("timezone_abbreviation", StringType()),
    StructField("elevation", DoubleType()),
    StructField("marketName", StringType()),
    StructField("daily_units", StructType([
        StructField("time", StringType()),
        StructField("temperature_2m_max", StringType()),
        StructField("temperature_2m_min", StringType()),
        StructField("rain_sum", StringType())
    ])),
    StructField("daily", StructType([
        StructField("time", ArrayType(StringType())),
        StructField("temperature_2m_max", ArrayType(DoubleType())),
        StructField("temperature_2m_min", ArrayType(DoubleType())),
        StructField("rain_sum", ArrayType(DoubleType()))
    ]))
])

weatherDataSparkDF = (
    df
    .withColumn("parsed", from_json(col("json"), schema))
    .select("parsed.*")
)

weatherDataSparkDF.display()

# (weatherDataSparkDF
# .write
# .mode("overwrite")
# .json(weatherDataSinkFolderPath))
  